In [1]:
import os
import json
import chromadb
from chromadb.utils import embedding_functions
from typing import TypedDict, List
from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, END
from fastapi import FastAPI
import uvicorn

# ==========================================
# Task 1: Create Corpus Documents & Embed in ChromaDB
# ==========================================
os.makedirs("docs", exist_ok=True)

documents = {
    "doc_01.txt": "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
    "doc_02.txt": "Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.",
    "doc_03.txt": "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.",
    "doc_04.txt": "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.",
    "doc_05.txt": "Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.",
    "doc_06.txt": "If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the in-app chat support with photos of the damaged item. Replacement or full refund will be processed immediately upon verification.",
    "doc_07.txt": "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.",
    "doc_08.txt": "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."
}

# Write text files into /docs folder
for filename, text in documents.items():
    with open(os.path.join("docs", filename), "w") as f:
        f.write(text)

# Initialize ChromaDB Vector Store with local embedding model
chroma_client = chromadb.Client()
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

collection = chroma_client.get_or_create_collection(
    name="zepto_policies",
    embedding_function=embedding_fn
)

# Populate ChromaDB collection
for doc_id, text in documents.items():
    collection.add(
        documents=[text],
        ids=[doc_id]
    )

print("=== Task 1 Complete: 8 Documents Indexed into ChromaDB ===")

# ==========================================
# Task 4: Pydantic Response Schema
# ==========================================
class SupportResponse(BaseModel):
    answer: str
    sources: List[str]
    confidence: float

# ==========================================
# Task 3: LangGraph Setup & Routing Logic
# ==========================================
class AgentState(TypedDict):
    query: str
    intent: str
    context: str
    sources: List[str]
    response: SupportResponse

# Node 1: Intent Classification
def classify_intent_node(state: AgentState):
    query = state['query'].lower()
    policy_keywords = ["delivery", "return", "refund", "membership", "tracking", "cancel", "gift card", "support hours"]
    
    if any(keyword in query for keyword in policy_keywords):
        intent = "policy_question"
    else:
        intent = "general_question"
        
    return {"intent": intent}

# Node 2: Policy Retrieval & Canned Answer Generation
def retrieve_and_answer_node(state: AgentState):
    results = collection.query(query_texts=[state['query']], n_results=3)
    top_doc_id = results['ids'][0][0]
    top_doc_text = results['documents'][0][0]
    
    snippet = top_doc_text[:200]
    answer_text = f"Based on the retrieved context: {snippet}"
    
    response = SupportResponse(
        answer=answer_text,
        sources=[top_doc_id],
        confidence=1.0
    )
    return {"context": top_doc_text, "sources": [top_doc_id], "response": response}

# Node 3: Direct General Answer Node
def direct_answer_node(state: AgentState):
    response = SupportResponse(
        answer="I can only answer questions about Zepto policies right now.",
        sources=[],
        confidence=1.0
    )
    return {"sources": [], "response": response}

# Conditional Routing Function
def route_intent(state: AgentState):
    if state['intent'] == "policy_question":
        return "retrieve_and_answer"
    return "direct_answer"

# Build LangGraph StateGraph
builder = StateGraph(AgentState)
builder.add_node("classify_intent", classify_intent_node)
builder.add_node("retrieve_and_answer", retrieve_and_answer_node)
builder.add_node("direct_answer", direct_answer_node)

builder.set_entry_point("classify_intent")
builder.add_conditional_edges("classify_intent", route_intent)
builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)

graph = builder.compile()

# ==========================================
# Test Calls (Verify Pipeline)
# ==========================================
print("\n=== Test 1: Policy Query (Trigger Retrieval) ===")
res1 = graph.invoke({"query": "What is the return policy for damaged grocery?"})
print(json.dumps(res1['response'].model_dump(), indent=2))

print("\n=== Test 2: General Query (Trigger Direct Answer) ===")
res2 = graph.invoke({"query": "What is the capital of France?"})
print(json.dumps(res2['response'].model_dump(), indent=2))

c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\HP\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\HP\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python 

=== Task 1 Complete: 8 Documents Indexed into ChromaDB ===

=== Test 1: Policy Query (Trigger Retrieval) ===
{
  "answer": "Based on the retrieved context: Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unop",
  "sources": [
    "doc_02.txt"
  ],
  "confidence": 1.0
}

=== Test 2: General Query (Trigger Direct Answer) ===
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}


In [2]:
# Task 5: FastAPI Application Setup
from fastapi import FastAPI
from pydantic import BaseModel

app = FastAPI(title="Zepto Support Assistant API")

class QueryRequest(BaseModel):
    query: str

@app.post("/ask", response_model=SupportResponse)
def ask_support(request: QueryRequest):
    result = graph.invoke({"query": request.query})
    return result['response']

print("=== Task 5 Complete: FastAPI '/ask' endpoint structure is ready ===")

=== Task 5 Complete: FastAPI '/ask' endpoint structure is ready ===
